In [1]:
import os
import json
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
DATA_DIR = "/kaggle/input/ml-challenge-udhgam-2"

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

train_data = load_jsonl(f"{DATA_DIR}/train.jsonl")
test_data  = load_jsonl(f"{DATA_DIR}/test.jsonl")

y = np.array([r["label"] for r in train_data])
N = len(train_data)

print("Train:", len(train_data), "Test:", len(test_data))
with open("/kaggle/input/udhgam-v4/folds.pkl", "rb") as f:
    folds = pickle.load(f)

N_FOLDS = len(folds)
print("Loaded folds:", N_FOLDS)
MAX_LEN = 256
BATCH_SIZE = 64
KERNEL_SIZES = [3, 5, 7]

EMBED_DIM = 128
NUM_FILTERS = 128
DROPOUT = 0.2
def compute_features(input_ids, attention_mask):
    """
    input_ids: List[int]
    attention_mask: List[int]
    Returns: np.array of shape (4,)
    """
    # Keep only real tokens
    tokens = [t for t, m in zip(input_ids, attention_mask) if m == 1]

    length = len(tokens)

    if length == 0:
        return np.zeros(4, dtype=np.float32)

    # 1. Normalized length
    norm_len = length / MAX_LEN

    # 2. Unique token ratio
    unique_ratio = len(set(tokens)) / length

    # 3. Token entropy
    counts = {}
    for t in tokens:
        counts[t] = counts.get(t, 0) + 1

    probs = np.array(list(counts.values()), dtype=np.float32) / length
    entropy = -np.sum(probs * np.log(probs + 1e-8))

    # Normalize entropy by log(length)
    entropy = entropy / np.log(length + 1e-8)

    # 4. Repetition ratio (adjacent)
    repeats = sum(tokens[i] == tokens[i+1] for i in range(length - 1))
    repetition_ratio = repeats / max(1, length - 1)

    return np.array(
        [norm_len, unique_ratio, entropy, repetition_ratio],
        dtype=np.float32
    )

all_ids = []

for row in train_data:
    all_ids.extend(row["input_ids"])

for row in test_data:
    all_ids.extend(row["input_ids"])

max_token_id = max(all_ids)
VOCAB_SIZE = max_token_id + 1

print("Max token id:", max_token_id)
print("Vocab size:", VOCAB_SIZE)

class PromptDataset(Dataset):
    def __init__(self, rows, indices, is_test=False):
        self.rows = rows
        self.indices = indices
        self.is_test = is_test

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        row = self.rows[self.indices[i]]

        input_ids = row["input_ids"][:MAX_LEN]
        attention = row["attention_mask"][:MAX_LEN]

        feats = compute_features(input_ids, attention)

        pad = MAX_LEN - len(input_ids)
        if pad > 0:
            input_ids += [0] * pad
            attention += [0] * pad

        item = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),

            "attention": torch.tensor(attention, dtype=torch.float),
            "features": torch.tensor(feats, dtype=torch.float),
        }

        if not self.is_test:
            item["label"] = torch.tensor(row["label"], dtype=torch.float)

        return item

Using device: cpu
Train: 79806 Test: 19952
Loaded folds: 5
Max token id: 50367
Vocab size: 50368


In [39]:
class AttnPoolCNNPromptModel(nn.Module):
    def __init__(self, vocab_size, num_features=4):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            EMBED_DIM,
            padding_idx=0
        )

        self.convs = nn.ModuleList([
            nn.Conv1d(EMBED_DIM, NUM_FILTERS, k, padding=k // 2)
            for k in KERNEL_SIZES
        ])

        # attention scorer (shared across kernels)
        self.attn_fc = nn.Linear(NUM_FILTERS, 1)

        self.dropout = nn.Dropout(DROPOUT)

        cnn_dim = len(KERNEL_SIZES) * NUM_FILTERS * 3  # attn + mean + max
        total_dim = cnn_dim + num_features

        self.regressor = nn.Sequential(
            nn.Linear(total_dim, 256),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(256, 1)
        )

    def attention_pool(self, x, mask):
        """
        x: (B, C, L)
        mask: (B, 1, L)
        """
        # (B, L, C)
        x_t = x.transpose(1, 2)

        scores = self.attn_fc(torch.tanh(x_t)).squeeze(-1)
        scores = scores.masked_fill(mask.squeeze(1) == 0, -1e9)

        attn = torch.softmax(scores, dim=1)
        pooled = torch.sum(x_t * attn.unsqueeze(-1), dim=1)

        return pooled
    def forward_features(self, input_ids, attention_mask, features):
        x = self.embedding(input_ids)
        x = x.transpose(1, 2)
        mask = attention_mask.unsqueeze(1)

        pooled = []

        for conv in self.convs:
            c = torch.relu(conv(x)) * mask

            attn_pool = self.attention_pool(c, mask)
            mean_pool = c.sum(dim=2) / (mask.sum(dim=2) + 1e-6)
            max_pool = c.max(dim=2).values

            pooled.extend([attn_pool, mean_pool, max_pool])

        cnn_features = torch.cat(pooled, dim=1)
        cnn_features = self.dropout(cnn_features)

        all_features = torch.cat([cnn_features, features], dim=1)
        return all_features
    

    def forward(self, input_ids, attention_mask, features):
        x = self.embedding(input_ids)          # (B, L, D)
        x = x.transpose(1, 2)                  # (B, D, L)
        mask = attention_mask.unsqueeze(1)

        pooled = []

        for conv in self.convs:
            c = torch.relu(conv(x)) * mask

            attn_pool = self.attention_pool(c, mask)
            mean_pool = c.sum(dim=2) / (mask.sum(dim=2) + 1e-6)
            max_pool = c.max(dim=2).values

            pooled.extend([attn_pool, mean_pool, max_pool])

        cnn_features = torch.cat(pooled, dim=1)
        cnn_features = self.dropout(cnn_features)

        all_features = torch.cat([cnn_features, features], dim=1)

        out = self.regressor(all_features).squeeze(1)
        return torch.sigmoid(out)

class AttnGRUPromptModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_features=4):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )

        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # scalar attention per timestep
        self.attn_fc = nn.Linear(2 * hidden_dim, 1)

        self.dropout = nn.Dropout(0.3)

        total_dim = 2 * hidden_dim + num_features

        self.regressor = nn.Sequential(
            nn.Linear(total_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, input_ids, attention_mask, features):
        x = self.embedding(input_ids)                # (B, L, D)

        lengths = attention_mask.sum(dim=1).long().cpu()

        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths, batch_first=True, enforce_sorted=False
        )

        packed_out, _ = self.gru(packed)

        out, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=MAX_LEN
        )                                            # (B, L, 2H)

        mask = attention_mask.unsqueeze(-1)

        # ---------- attention pooling ----------
        scores = self.attn_fc(torch.tanh(out)).squeeze(-1)  # (B, L)
        scores = scores.masked_fill(attention_mask == 0, -1e9)

        attn = torch.softmax(scores, dim=1)
        attn_pool = torch.sum(out * attn.unsqueeze(-1), dim=1)

        # ---------- stability fallback ----------
        mean_pool = (out * mask).sum(dim=1) / (mask.sum(dim=1) + 1e-6)

        rep = 0.7 * attn_pool + 0.3 * mean_pool
        rep = self.dropout(rep)

        out = self.regressor(torch.cat([rep, features], dim=1)).squeeze(1)
        return torch.sigmoid(out)


class SegmentedGRUPromptModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_features=4):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )

        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.2)

        # 3 segments × 2*hidden_dim
        total_dim = 3 * (2 * hidden_dim) + num_features

        self.regressor = nn.Sequential(
            nn.Linear(total_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, input_ids, attention_mask, features):
        x = self.embedding(input_ids)  # (B, L, D)

        lengths = attention_mask.sum(dim=1).long().cpu()

        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths, batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.gru(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=MAX_LEN
        )  # (B, L, 2H)

        mask = attention_mask.unsqueeze(-1)
        out = out * mask

        reps = []
        for i, L in enumerate(lengths):
            L = int(L.item())
            if L < 3:
                pooled = out[i, :L].mean(dim=0)
                reps.append(torch.cat([pooled, pooled, pooled]))
            else:
                s1 = out[i, :L//3].mean(dim=0)
                s2 = out[i, L//3:2*L//3].mean(dim=0)
                s3 = out[i, 2*L//3:L].mean(dim=0)
                reps.append(torch.cat([s1, s2, s3]))

        gru_features = torch.stack(reps)
        gru_features = self.dropout(gru_features)

        all_features = torch.cat([gru_features, features], dim=1)
        out = self.regressor(all_features).squeeze(1)

        return torch.sigmoid(out)


class CNNPromptModel(nn.Module):
    def __init__(self, vocab_size=50000, num_features=4):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=EMBED_DIM,
            padding_idx=0
        )

        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=EMBED_DIM,
                out_channels=NUM_FILTERS,
                kernel_size=k,
                padding=k // 2
            )
            for k in KERNEL_SIZES
        ])

        self.dropout = nn.Dropout(DROPOUT)

        cnn_out_dim = len(KERNEL_SIZES) * NUM_FILTERS * 2
        total_dim = cnn_out_dim + num_features

        self.regressor = nn.Sequential(
            nn.Linear(total_dim, 128),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(128, 1)
        )

    def forward(self, input_ids, attention_mask, features):
        x = self.embedding(input_ids)
        x = x.transpose(1, 2)

        pooled_outputs = []

        for conv in self.convs:
            c = torch.relu(conv(x))
            mask = attention_mask.unsqueeze(1)
            c = c * mask

            max_pool = torch.max(c, dim=2).values
            mean_pool = torch.sum(c, dim=2) / (mask.sum(dim=2) + 1e-6)

            pooled_outputs.append(max_pool)
            pooled_outputs.append(mean_pool)

        cnn_features = torch.cat(pooled_outputs, dim=1)
        cnn_features = self.dropout(cnn_features)

        all_features = torch.cat([cnn_features, features], dim=1)

        out = self.regressor(all_features).squeeze(1)
        return torch.sigmoid(out)
class GRUPromptModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_features=4):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )

        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.2)

        # GRU output: 2 * hidden_dim (bi)
        total_dim = 2 * hidden_dim + num_features

        self.regressor = nn.Sequential(
            nn.Linear(total_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, input_ids, attention_mask, features):
        x = self.embedding(input_ids)          # (B, L, D)

        lengths = attention_mask.sum(dim=1).long().cpu()

        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths, batch_first=True, enforce_sorted=False
        )

        packed_out, _ = self.gru(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=MAX_LEN
        )                                       # (B, L, 2H)

        mask = attention_mask.unsqueeze(-1)

        out = out * mask

        mean_pool = out.sum(dim=1) / (mask.sum(dim=1) + 1e-6)
        max_pool = out.max(dim=1).values

        gru_features = 0.5 * (mean_pool + max_pool)
        gru_features = self.dropout(gru_features)

        all_features = torch.cat([gru_features, features], dim=1)

        out = self.regressor(all_features).squeeze(1)
        return torch.sigmoid(out)
# ===============================
# CNN kernel sizes (must match training)
# ===============================

KERNEL_SIZES = [3, 5, 7]

class DilatedCNNPromptModel(nn.Module):
    def __init__(self, vocab_size, num_features=4):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=EMBED_DIM,
            padding_idx=0
        )

        # Dilated convolutions
        self.convs = nn.ModuleList([
            nn.Conv1d(EMBED_DIM, NUM_FILTERS, kernel_size=3, dilation=1, padding=1),
            nn.Conv1d(EMBED_DIM, NUM_FILTERS, kernel_size=3, dilation=2, padding=2),
            nn.Conv1d(EMBED_DIM, NUM_FILTERS, kernel_size=3, dilation=4, padding=4),
            nn.Conv1d(EMBED_DIM, NUM_FILTERS, kernel_size=3, dilation=8, padding=8),
        ])

        self.dropout = nn.Dropout(DROPOUT)

        cnn_out_dim = len(self.convs) * NUM_FILTERS * 2
        total_dim = cnn_out_dim + num_features

        self.regressor = nn.Sequential(
            nn.Linear(total_dim, 128),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(128, 1)
        )

    def forward(self, input_ids, attention_mask, features):
        x = self.embedding(input_ids)        # (B, L, D)
        x = x.transpose(1, 2)                # (B, D, L)

        pooled = []
        mask = attention_mask.unsqueeze(1)

        for conv in self.convs:
            c = torch.relu(conv(x))
            c = c * mask

            max_pool = c.max(dim=2).values
            mean_pool = c.sum(dim=2) / (mask.sum(dim=2) + 1e-6)

            pooled.append(max_pool)
            pooled.append(mean_pool)

        cnn_features = torch.cat(pooled, dim=1)
        cnn_features = self.dropout(cnn_features)

        all_features = torch.cat([cnn_features, features], dim=1)
        out = self.regressor(all_features).squeeze(1)

        return torch.sigmoid(out)

In [3]:
MODEL_ZOO = {
    "cnn": CNNPromptModel,
    "cnn_seed2": CNNPromptModel,
    "bigru": GRUPromptModel,
    "bigru_seed2": GRUPromptModel,
    "seg_gru": SegmentedGRUPromptModel,
    "attn_gru": AttnGRUPromptModel,
    "attn_cnn": AttnPoolCNNPromptModel,
    "dilated_cnn": DilatedCNNPromptModel,
}

oof = {k: np.zeros(N) for k in MODEL_ZOO}
test_preds = {k: [] for k in MODEL_ZOO}

test_idx = np.arange(len(test_data))
test_ds = PromptDataset(test_data, test_idx, is_test=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

for name, ModelClass in MODEL_ZOO.items():
    print(f"\n🔹 Loading {name}")
    fold_test_preds = []

    for f, (tr_idx, va_idx) in enumerate(folds):
        model = ModelClass(vocab_size=50368).to(DEVICE)
        ckpt = torch.load(f"/kaggle/input/udhgam-v4/checkpoints/{name}_fold{f}.pt", map_location=DEVICE)
        model.load_state_dict(ckpt)
        model.eval()

        # ---- OOF ----
        val_ds = PromptDataset(train_data, va_idx)
        val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

        preds = []
        with torch.no_grad():
            for b in val_loader:
                p = model(
                    b["input_ids"].to(DEVICE),
                    b["attention"].to(DEVICE),
                    b["features"].to(DEVICE),
                )
                preds.append(p.cpu().numpy())

        oof[name][va_idx] = np.concatenate(preds)

        # ---- TEST ----
        tp = []
        with torch.no_grad():
            for b in test_loader:
                p = model(
                    b["input_ids"].to(DEVICE),
                    b["attention"].to(DEVICE),
                    b["features"].to(DEVICE),
                )
                tp.append(p.cpu().numpy())

        fold_test_preds.append(np.concatenate(tp))

    test_preds[name] = np.mean(fold_test_preds, axis=0)
oof_df = pd.DataFrame(oof)
test_df = pd.DataFrame(test_preds)

# aggregate signals
for df in [oof_df, test_df]:
    df["mean"] = df.mean(axis=1)
    df["std"]  = df.std(axis=1)
    df["min"]  = df.min(axis=1)
    df["max"]  = df.max(axis=1)

X_oof = oof_df.values
X_test = test_df.values



🔹 Loading cnn

🔹 Loading cnn_seed2

🔹 Loading bigru

🔹 Loading bigru_seed2

🔹 Loading seg_gru

🔹 Loading attn_gru

🔹 Loading attn_cnn

🔹 Loading dilated_cnn


In [4]:
def add_disagreement_features(X):
    X = X.copy()

    X["pred_mean"] = X.mean(axis=1)
    X["pred_std"]  = X.std(axis=1)
    X["pred_max"]  = X.max(axis=1)
    X["pred_min"]  = X.min(axis=1)
    X["pred_range"] = X["pred_max"] - X["pred_min"]

    # how many models strongly disagree with the mean
    for t in [0.05, 0.1]:
        X[f"outlier_frac_{t}"] = (
            (np.abs(X.sub(X["pred_mean"], axis=0)) > t).mean(axis=1)
        )

    return X


In [5]:
X_oof_plus  = add_disagreement_features(pd.DataFrame(X_oof))
X_test_plus = add_disagreement_features(pd.DataFrame(X_test))


In [6]:
meta_lgb = LGBMRegressor(
    n_estimators=1800,
    learning_rate=0.015,
    max_depth=5,
    num_leaves=32,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=0.2,
    reg_lambda=0.5,
    objective="regression_l1",
    random_state=42,
    verbosity=-1
)

meta_lgb.fit(X_oof_plus, y)

meta_oof  = meta_lgb.predict(X_oof_plus)
meta_test = meta_lgb.predict(X_test_plus)

print("Meta LGBM + disagreement OOF MAE:",
      mean_absolute_error(y, meta_oof))


Meta LGBM + disagreement OOF MAE: 0.1473087322306548


In [7]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error
import pandas as pd
import numpy as np

def add_global_disagreement(df):
    df = df.copy()
    df["pred_mean"] = df.mean(axis=1)
    df["pred_std"]  = df.std(axis=1)
    df["pred_max"]  = df.max(axis=1)
    df["pred_min"]  = df.min(axis=1)
    return df

# keep originals intact
X_oof_df  = pd.DataFrame(X_oof)
X_test_df = pd.DataFrame(X_test)

X_oof_gd  = add_global_disagreement(X_oof_df)
X_test_gd = add_global_disagreement(X_test_df)

meta_lgb_gd = LGBMRegressor(
    n_estimators=1800,
    learning_rate=0.015,
    max_depth=5,
    num_leaves=32,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=0.2,
    reg_lambda=0.5,
    objective="regression_l1",
    random_state=42,
    verbosity=-1
)

meta_lgb_gd.fit(X_oof_gd, y)

meta_oof_gd  = meta_lgb_gd.predict(X_oof_gd)
meta_test_gd = meta_lgb_gd.predict(X_test_gd)

print("BLOCK 1 | Meta LGBM + global disagreement OOF MAE:",
      mean_absolute_error(y, meta_oof_gd))


BLOCK 1 | Meta LGBM + global disagreement OOF MAE: 0.14774450517172366


In [8]:
def add_per_model_deviation(df):
    df = df.copy()
    mean = df.mean(axis=1)
    for c in df.columns:
        df[f"{c}_dev"] = df[c] - mean
    return df

X_oof_dev  = add_per_model_deviation(X_oof_gd)
X_test_dev = add_per_model_deviation(X_test_gd)

meta_lgb_dev = LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.012,
    max_depth=5,
    num_leaves=32,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=0.3,
    reg_lambda=0.6,
    objective="regression_l1",
    random_state=42,
    verbosity=-1
)

meta_lgb_dev.fit(X_oof_dev, y)

meta_oof_dev  = meta_lgb_dev.predict(X_oof_dev)
meta_test_dev = meta_lgb_dev.predict(X_test_dev)

print("BLOCK 2 | Meta LGBM + global + per-model deviation OOF MAE:",
      mean_absolute_error(y, meta_oof_dev))


BLOCK 2 | Meta LGBM + global + per-model deviation OOF MAE: 0.14717906476836876


In [9]:
eps = 1e-4
y_logit = np.log((y + eps) / (1 - y + eps))

meta_lgb_logit = LGBMRegressor(
    n_estimators=2200,
    learning_rate=0.01,
    max_depth=5,
    num_leaves=32,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=0.3,
    reg_lambda=0.7,
    objective="regression_l1",
    random_state=42,
    verbosity=-1
)

meta_lgb_logit.fit(X_oof_dev, y_logit)

meta_oof_logit = meta_lgb_logit.predict(X_oof_dev)
meta_test_logit = meta_lgb_logit.predict(X_test_dev)

# inverse logit
meta_oof_logit = 1 / (1 + np.exp(-meta_oof_logit))
meta_test_logit = 1 / (1 + np.exp(-meta_test_logit))

print("BLOCK 3 | Meta LGBM + logit target + disagreement OOF MAE:",
      mean_absolute_error(y, meta_oof_logit))


BLOCK 3 | Meta LGBM + logit target + disagreement OOF MAE: 0.14627666602568004


In [10]:
# ============================
# FINAL SUBMISSION GENERATION (FIXED)
# ============================

# Safety checks
assert len(meta_test_logit) == len(test_data), "Prediction length mismatch!"

# Clip predictions for safety
meta_test_logit = np.clip(meta_test_logit, 0.0, 1.0)

# Generate correct example_id format
example_ids = [f"te_{i:07d}" for i in range(len(meta_test_logit))]

submission = pd.DataFrame({
    "example_id": example_ids,
    "label": meta_test_logit
})

SUB_PATH = "submissionMETAv2.csv"
submission.to_csv(SUB_PATH, index=False)

print("✅ Submission saved:", SUB_PATH)
print(submission.head())
print(submission.tail())
##score 0.15221

✅ Submission saved: submissionMETAv2.csv
   example_id     label
0  te_0000000  0.308095
1  te_0000001  0.913416
2  te_0000002  0.581268
3  te_0000003  0.337766
4  te_0000004  0.206169
       example_id     label
19947  te_0019947  0.661185
19948  te_0019948  0.170715
19949  te_0019949  0.636904
19950  te_0019950  0.340454
19951  te_0019951  0.179537


In [11]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

meta_lgb_def = LGBMRegressor(
    n_estimators=1500,      # ↓ trees
    learning_rate=0.015,
    max_depth=4,            # ↓ depth
    num_leaves=24,          # ↓ leaves
    subsample=0.65,
    colsample_bytree=0.65,
    reg_alpha=0.4,          # ↑ regularization
    reg_lambda=0.8,
    objective="regression_l1",
    random_state=42,
    verbosity=-1
)

# fit on SAME features you used for logit model
meta_lgb_def.fit(X_oof_dev, y)

meta_oof_def  = meta_lgb_def.predict(X_oof_dev)
meta_test_def = meta_lgb_def.predict(X_test_dev)

print(
    "Defensive Meta OOF MAE:",
    mean_absolute_error(y, meta_oof_def)
)


Defensive Meta OOF MAE: 0.1495254268132925


In [12]:
# Safety clip
meta_test_logit = np.clip(meta_test_logit, 0.0, 1.0)
meta_test_def   = np.clip(meta_test_def,   0.0, 1.0)

# FINAL BLEND (do NOT change weights initially)
final_test_pred = (
    0.7 * meta_test_logit +
    0.3 * meta_test_def
)

final_test_pred = np.clip(final_test_pred, 0.0, 1.0)


In [13]:
# load sample submission to guarantee format
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

assert len(sample_sub) == len(final_test_pred)

submission = pd.DataFrame({
    "example_id": sample_sub["example_id"],
    "label": final_test_pred
})

SUB_PATH = "submission_final_defensive_blend.csv"
submission.to_csv(SUB_PATH, index=False)

print("✅ Submission saved:", SUB_PATH)
print(submission.head())
print(submission.describe())
## score 0.15218(my best submission)

✅ Submission saved: submission_final_defensive_blend.csv
   example_id     label
0  te_0000000  0.309995
1  te_0000001  0.912650
2  te_0000002  0.578149
3  te_0000003  0.336061
4  te_0000004  0.206690
              label
count  19952.000000
mean       0.532703
std        0.282930
min        0.006155
25%        0.284681
50%        0.580451
75%        0.786774
max        0.962001


In [14]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

ridge = Ridge(
    alpha=10.0,      # STRONG regularization
    fit_intercept=True,
    random_state=42
)

ridge.fit(X_oof, y)

ridge_oof  = ridge.predict(X_oof)
ridge_test = ridge.predict(X_test)

ridge_oof  = np.clip(ridge_oof, 0, 1)
ridge_test = np.clip(ridge_test, 0, 1)

print("Ridge OOF MAE:", mean_absolute_error(y, ridge_oof))


Ridge OOF MAE: 0.15970719784064458


In [15]:
from sklearn.linear_model import ElasticNet

enet = ElasticNet(
    alpha=5.0,
    l1_ratio=0.1,   # mostly ridge
    max_iter=5000,
    random_state=42
)

enet.fit(X_oof, y)

enet_oof  = enet.predict(X_oof)
enet_test = enet.predict(X_test)

enet_oof  = np.clip(enet_oof, 0, 1)
enet_test = np.clip(enet_test, 0, 1)

print("ElasticNet OOF MAE:", mean_absolute_error(y, enet_oof))


ElasticNet OOF MAE: 0.29179793115177544


In [16]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

ridge = Ridge(
    alpha=2.0,
    fit_intercept=True,
    random_state=42
)

ridge.fit(X_oof, y)

ridge_oof  = ridge.predict(X_oof)
ridge_test = ridge.predict(X_test)

ridge_oof  = np.clip(ridge_oof, 0, 1)
ridge_test = np.clip(ridge_test, 0, 1)

print("Ridge OOF MAE:", mean_absolute_error(y, ridge_oof))


Ridge OOF MAE: 0.15969591541115805


In [17]:
mean_test = test_df["mean"].values
mean_oof  = oof_df["mean"].values

print("Mean OOF MAE:", mean_absolute_error(y, mean_oof))


Mean OOF MAE: 0.15334892129579072


In [18]:
final_test_pred = (
    0.55 * mean_test +
    0.25 * ridge_test +
    0.20 * meta_test_logit
)

final_test_pred = np.clip(final_test_pred, 0.0, 1.0)


In [19]:
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

submission = pd.DataFrame({
    "example_id": sample_sub["example_id"],
    "label": final_test_pred
})

submission.to_csv("submission_FINAL_LOWVAR_BLEND.csv", index=False)
print("✅ Final submission written")
## score 0.15310

✅ Final submission written


In [20]:
# =========================
# Global setup
# =========================
from collections import Counter, defaultdict
import numpy as np
import zlib

MAX_LEN = 256

GLOBAL_CNT = Counter(t for row in train_data for t in row["input_ids"])
TOTAL_TOKENS = sum(GLOBAL_CNT.values())

TOP_COMMON_TOKENS = set(t for t, _ in GLOBAL_CNT.most_common(50))
COMMON_TOKENS = TOP_COMMON_TOKENS

TOP_K = 200
TOP_K_TOKENS = [t for t, _ in GLOBAL_CNT.most_common(TOP_K)]

RARE_Q = [0.001, 0.01, 0.1]

FEATURE_DIM = 34 + TOP_K + 8  # keep in sync


def extract_features(row):
    tokens = [t for t, m in zip(row["input_ids"], row["attention_mask"]) if m == 1]
    L = len(tokens)

    if L == 0:
        return np.zeros(FEATURE_DIM, dtype=np.float32)

    feats = []

    # =====================================================
    # 1. Length & density
    # =====================================================
    uniq = len(set(tokens))
    feats += [
        L,
        np.log1p(L),
        uniq / L,
        L / max(uniq, 1),
        1 - L / MAX_LEN
    ]

    # =====================================================
    # 2. Global entropy & repetition
    # =====================================================
    cnt = Counter(tokens)
    freqs = np.array(list(cnt.values())) / L

    feats += [
        -np.sum(freqs * np.log(freqs + 1e-8)) / np.log(L + 1e-8),
        np.sum(freqs ** 2),
        np.max(freqs),
        np.sum(np.sort(freqs)[-3:])
    ]

    feats.append(
        np.mean([tokens[i] == tokens[i+1] for i in range(L-1)]) if L > 1 else 0
    )

    # =====================================================
    # 3. Positional identity (hashed)
    # =====================================================
    def hash_id(t, mod=50):
        return (t % mod) / mod

    lead = tokens[:3] if L >= 3 else tokens
    tail = tokens[-3:] if L >= 3 else tokens

    feats += [hash_id(t) for t in lead]
    feats += [hash_id(t) for t in tail]

    # =====================================================
    # 4. Token recurrence distance
    # =====================================================
    pos = defaultdict(list)
    for i, t in enumerate(tokens):
        pos[t].append(i)

    gaps = [d for p in pos.values() if len(p) > 1 for d in np.diff(p)]
    feats += [np.mean(gaps), np.std(gaps), np.max(gaps)] if gaps else [0, 0, 0]

    # =====================================================
    # 5. Structural anchor variance
    # =====================================================
    anchor_std = []
    for tok, _ in cnt.most_common(5):
        p = pos[tok]
        if len(p) > 2:
            anchor_std.append(np.std(np.diff(p)))
    feats.append(np.mean(anchor_std) if anchor_std else 0)

    # =====================================================
    # 6. N-gram repetition
    # =====================================================
    for n in [2, 3]:
        ngrams = [tuple(tokens[i:i+n]) for i in range(L-n+1)]
        c = Counter(ngrams)
        feats += [
            max(c.values()) / len(ngrams),
            sum(v > 1 for v in c.values()) / len(c)
        ] if c else [0, 0]

    # =====================================================
    # 7. Bigram entropy
    # =====================================================
    if L > 2:
        bigrams = [(tokens[i], tokens[i+1]) for i in range(L-1)]
        bc = Counter(bigrams)
        p = np.array(list(bc.values())) / len(bigrams)
        feats.append(-np.sum(p * np.log(p + 1e-8)))
    else:
        feats.append(0)

    # =====================================================
    # 8. Rarity profile
    # =====================================================
    global_freqs = np.array([GLOBAL_CNT[t] / TOTAL_TOKENS for t in tokens])

    feats += [
        np.mean(global_freqs < RARE_Q[0]),
        np.mean((global_freqs >= RARE_Q[0]) & (global_freqs < RARE_Q[1])),
        np.mean((global_freqs >= RARE_Q[1]) & (global_freqs < RARE_Q[2])),
        np.mean(global_freqs >= RARE_Q[2]),
        np.mean(global_freqs),
        np.std(global_freqs),
        np.min(global_freqs)
    ]

    # =====================================================
    # 9. Compression ratio
    # =====================================================
    raw = np.array(tokens, dtype=np.uint32).tobytes()
    feats.append(len(zlib.compress(raw)) / max(len(raw), 1))

    # =====================================================
    # 10. Top-K identity
    # =====================================================
    token_set = set(tokens)
    feats += [1.0 if t in token_set else 0.0 for t in TOP_K_TOKENS]

    # =====================================================
    # 11. Delimiter detection
    # =====================================================
    feats.append(sum((t in COMMON_TOKENS) and (cnt[t] <= 3) for t in cnt))

    # =====================================================
    # 12. 🔥 WINDOW ENTROPY (PHASE SHIFT SIGNAL)
    # =====================================================
    def window_entropy(seg):
        if len(seg) < 2:
            return 0
        c = Counter(seg)
        p = np.array(list(c.values())) / len(seg)
        return -np.sum(p * np.log(p + 1e-8)) / np.log(len(seg) + 1e-8)

    q1 = L // 4
    q2 = L // 2
    q3 = 3 * L // 4

    e1 = window_entropy(tokens[:q1])
    e2 = window_entropy(tokens[q1:q2])
    e3 = window_entropy(tokens[q2:q3])
    e4 = window_entropy(tokens[q3:])

    feats += [
        e1, e2, e3, e4,
        e2 - e1,
        e3 - e2,
        e4 - e3,
        max(e1, e2, e3, e4) - min(e1, e2, e3, e4)
    ]

    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)
    return np.array(feats, dtype=np.float32)


In [21]:
from sklearn.metrics import mean_absolute_error
from lightgbm import LGBMRegressor
import numpy as np

meta_oof_stack = np.zeros(len(y))
meta_test_stack = []

for f, (_, val_idx) in enumerate(folds):
    print(f"\n🧠 Training META for Fold {f}")

    train_idx = np.setdiff1d(np.arange(len(y)), val_idx)

    X_tr = X_oof_dev.iloc[train_idx]
    y_tr = y_logit[train_idx]

    X_va = X_oof_dev.iloc[val_idx]

    meta = LGBMRegressor(
        n_estimators=2000,
        learning_rate=0.01,
        max_depth=5,
        num_leaves=32,
        subsample=0.7,
        colsample_bytree=0.7,
        reg_alpha=0.3,
        reg_lambda=0.7,
        objective="regression_l1",
        random_state=42 + f,
        verbosity=-1
    )

    meta.fit(X_tr, y_tr)

    # ----- OOF -----
    oof_pred = meta.predict(X_va)
    oof_pred = 1 / (1 + np.exp(-oof_pred))
    meta_oof_stack[val_idx] = oof_pred

    # ----- TEST -----
    test_pred = meta.predict(X_test_dev)
    test_pred = 1 / (1 + np.exp(-test_pred))
    meta_test_stack.append(test_pred)

print(
    "🔥 STACKED META OOF MAE:",
    mean_absolute_error(y, meta_oof_stack)
)



🧠 Training META for Fold 0

🧠 Training META for Fold 1

🧠 Training META for Fold 2

🧠 Training META for Fold 3

🧠 Training META for Fold 4
🔥 STACKED META OOF MAE: 0.1529163309543612


In [25]:
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import KFold
import numpy as np

cal_oof = np.zeros_like(y)
cal_test_preds = []

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for tr, va in kf.split(y):
    iso = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip")
    iso.fit(meta_oof_logit[tr], y[tr])

    cal_oof[va] = iso.predict(meta_oof_logit[va])
    cal_test_preds.append(iso.predict(meta_test_logit))

print(
    "🔥 CALIBRATED OOF MAE:",
    mean_absolute_error(y, cal_oof)
)

cal_test = np.mean(cal_test_preds, axis=0)


🔥 CALIBRATED OOF MAE: 0.15286411397636462


In [26]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_safe = np.zeros(len(y))

for tr, va in kf.split(X_oof_dev):
    m = LGBMRegressor(
        objective="quantile",
        alpha=0.5,
        n_estimators=2200,
        learning_rate=0.03,
        max_depth=7,
        num_leaves=45,
        subsample=0.7,
        colsample_bytree=0.7,
        reg_alpha=0.3,
        reg_lambda=0.7,
        random_state=42,
        verbosity=-1
    )

    m.fit(X_oof_dev.iloc[tr], y_logit[tr])
    p = m.predict(X_oof_dev.iloc[va])
    p = 1 / (1 + np.exp(-p))
    oof_safe[va] = p

print("SAFE CV MAE:", mean_absolute_error(y, oof_safe))


SAFE CV MAE: 0.15377187428149305


In [27]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import numpy as np
from lightgbm import LGBMRegressor

def safe_logit_meta_cv(
    X, y, params, n_splits=5, seed=42
):
    eps = 1e-4
    y_logit = np.log((y + eps) / (1 - y + eps))

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(y))

    for fold, (tr, va) in enumerate(kf.split(X)):
        model = LGBMRegressor(**params)

        model.fit(
            X.iloc[tr],
            y_logit[tr]
        )

        p = model.predict(X.iloc[va])
        p = 1 / (1 + np.exp(-p))   # inverse logit
        oof[va] = p

    mae = mean_absolute_error(y, oof)
    return mae


In [28]:
base_params = dict(
    objective="regression_l1",
    n_estimators=2200,
    learning_rate=0.002,
    max_depth=4,
    num_leaves=16,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=0.3,
    reg_lambda=0.7,
    random_state=42,
    verbosity=-1
)

mae = safe_logit_meta_cv(X_oof_dev, y, base_params)
print("SAFE CV MAE:", mae)


SAFE CV MAE: 0.15264864780679996


In [29]:
eps = 1e-4
y_logit = np.log((y + eps) / (1 - y + eps))

final_meta = LGBMRegressor(
    objective="regression_l1",   # or quantile if you prefer that variant
    n_estimators=2200,
    learning_rate=0.002,
    max_depth=4,
    num_leaves=16,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=0.3,
    reg_lambda=0.7,
    random_state=42,
    verbosity=-1
)

final_meta.fit(X_oof_dev, y_logit)

final_test_logit = final_meta.predict(X_test_dev)
final_test = 1 / (1 + np.exp(-final_test_logit))
final_test = np.clip(final_test, 0.0, 1.0)


In [30]:
# -----------------------------
# WRITE SUBMISSION
# -----------------------------
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

assert len(sample_sub) == len(final_test), "Length mismatch!"

submission = pd.DataFrame({
    "example_id": sample_sub["example_id"],
    "label": final_test
})

SUB_PATH = "submission_FINAL_META_FULLDATA.csv"
submission.to_csv(SUB_PATH, index=False)

print("✅ Submission written:", SUB_PATH)
print(submission.head())
print(submission.describe())


✅ Submission written: submission_FINAL_META_FULLDATA.csv
   example_id     label
0  te_0000000  0.316761
1  te_0000001  0.911832
2  te_0000002  0.582159
3  te_0000003  0.331804
4  te_0000004  0.212431
              label
count  19952.000000
mean       0.532325
std        0.281776
min        0.008721
25%        0.284847
50%        0.581147
75%        0.784153
max        0.950195


In [31]:
from collections import defaultdict
def relative_position_variance(row):
    tokens = [t for t, m in zip(row["input_ids"], row["attention_mask"]) if m]
    L = len(tokens)
    pos = defaultdict(list)
    for i, t in enumerate(tokens):
        pos[t].append(i / L)
    vars = [np.var(v) for v in pos.values() if len(v) >= 2]
    return np.mean(vars) if vars else 1.0
rpc = np.array([relative_position_variance(r) for r in train_data])
print(np.corrcoef(rpc, y)[0,1])

bucket_hits = defaultdict(int)

for r in train_data:
    tokens = r["input_ids"]
    L = len(tokens)
    for i, t in enumerate(tokens):
        rp = i / L
        if abs(rp - 0.25) < 0.02 or abs(rp - 0.5) < 0.02 or abs(rp - 0.75) < 0.02:
            bucket_hits[t] += 1

top_boundary_tokens = sorted(bucket_hits.items(), key=lambda x: -x[1])[:50]
# Create a set of the top tokens for fast lookup
top_tokens_set = {t for t, count in top_boundary_tokens}

counts_per_prompt = []
y_values = []

for r in train_data:
    tokens = r["input_ids"]
    # Count how many tokens in this prompt are in our 'top_boundary' list
    hit_count = sum(1 for t in tokens if t in top_tokens_set)
    
    counts_per_prompt.append(hit_count)
    y_values.append(r["label"]) # Assuming 'y' is a key in your dict

import numpy as np
from scipy.stats import pearsonr

# Calculate density (hits per token) to remove length bias
x_density = np.array([count / len(r["input_ids"]) for count, r in zip(counts_per_prompt, train_data)])
y = np.array(y_values)

correlation, p_value = pearsonr(x_density, y)

print(f"Pearson Correlation (Density): {correlation:.4f}")
print(f"P-value: {p_value:.4e}")

from collections import Counter, defaultdict
import numpy as np
import zlib

MAX_LEN = 256

GLOBAL_CNT = Counter(t for row in train_data for t in row["input_ids"])

global_top = [t for t,_ in GLOBAL_CNT.most_common(200)]

def directive_density(row):
    tokens = row["input_ids"]
    L = len(tokens)
    head = tokens[:max(10, L//5)]
    return sum(t in global_top for t in head) / len(head)
# 1. Calculate the directive_density for every row in train_data
# We use the 'y' array we already created from row["label"]
densities = np.array([directive_density(r) for r in train_data])

# 2. Calculate Correlation
from scipy.stats import pearsonr

corr, p_val = pearsonr(densities, y)

print(f"Directive Density Correlation with Label: {corr:.4f}")
print(f"P-value: {p_val:.4e}")
def constraint_score(row):
    tokens = [t for t,m in zip(row["input_ids"], row["attention_mask"]) if m]
    cnt = Counter(tokens)
    return sum(v >= 3 for v in cnt.values())
# 1. Calculate scores for all rows
# Note: Using the 'y' array we defined earlier as np.array([r["label"] for r in train_data])
scores = np.array([constraint_score(r) for r in train_data])

# 2. Compute Pearson correlation
from scipy.stats import pearsonr

corr, p_val = pearsonr(scores, y)

print(f"Constraint Score Correlation: {corr:.4f}")
print(f"P-value: {p_val:.4e}")
def structural_features(row):
    tokens = [t for t,m in zip(row["input_ids"], row["attention_mask"]) if m]
    L = len(tokens)

    # RPC
    pos = defaultdict(list)
    for i, t in enumerate(tokens):
        pos[t].append(i / L)
    rpc = np.mean([np.var(v) for v in pos.values() if len(v) >= 2]) if L else 1.0

    # Constraint score
    cnt = Counter(tokens)
    constraint = sum(v >= 3 for v in cnt.values())

    # Boundary density
    hits = 0
    for i,t in enumerate(tokens):
        rp = i / L
        if abs(rp-0.25)<0.02 or abs(rp-0.5)<0.02 or abs(rp-0.75)<0.02:
            if t in top_tokens_set:
                hits += 1
    boundary_density = hits / L

    # Length (raw, not normalized)
    length = L

    return np.array([rpc, constraint, boundary_density, length], dtype=np.float32)


-0.3422664286500887
Pearson Correlation (Density): 0.2003
P-value: 0.0000e+00
Directive Density Correlation with Label: -0.0336
P-value: 2.4029e-21
Constraint Score Correlation: 0.3466
P-value: 0.0000e+00


In [32]:
X_struct = np.vstack([structural_features(r) for r in train_data])
X_struct_test = np.vstack([structural_features(r) for r in test_data])

X_meta_final = np.hstack([X_oof_dev.values, X_struct])
X_meta_test_final = np.hstack([X_test_dev.values, X_struct_test])


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [33]:
eps = 1e-4
y_logit = np.log((y + eps) / (1 - y + eps))

meta = LGBMRegressor(
    objective="regression_l1",
    n_estimators=1800,
    learning_rate=0.01,
    max_depth=4,
    num_leaves=24,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=0.3,
    reg_lambda=0.7,
    random_state=42,
    verbosity=-1
)

meta.fit(X_meta_final, y_logit)

oof = 1 / (1 + np.exp(-meta.predict(X_meta_final)))
print("🔥 OOF MAE:", mean_absolute_error(y, oof))


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


🔥 OOF MAE: 0.1490348528341288


In [40]:
from sklearn.metrics import mean_absolute_error

EMB_DIM = None  # will infer dynamically

train_embs = np.zeros((len(train_data), 1))  # temp
train_preds = np.zeros(len(train_data))

for f, (tr_idx, va_idx) in enumerate(folds):
    print(f"🔹 Fold {f}")

    model = AttnPoolCNNPromptModel(vocab_size=VOCAB_SIZE).to(DEVICE)
    ckpt = torch.load(f"/kaggle/input/udhgam-v4/checkpoints/attn_cnn_fold{f}.pt",
                      map_location=DEVICE)
    model.load_state_dict(ckpt)
    model.eval()

    val_ds = PromptDataset(train_data, va_idx)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

    emb_list, pred_list = [], []

    with torch.no_grad():
        for b in val_loader:
            feats = model.forward_features(
                b["input_ids"].to(DEVICE),
                b["attention"].to(DEVICE),
                b["features"].to(DEVICE),
            )
            out = model(
                b["input_ids"].to(DEVICE),
                b["attention"].to(DEVICE),
                b["features"].to(DEVICE),
            )

            emb_list.append(feats.cpu().numpy())
            pred_list.append(out.cpu().numpy())

    emb = np.vstack(emb_list)
    preds = np.concatenate(pred_list)

    if EMB_DIM is None:
        EMB_DIM = emb.shape[1]
        train_embs = np.zeros((len(train_data), EMB_DIM))

    train_embs[va_idx] = emb
    train_preds[va_idx] = preds

print("Base OOF MAE:", mean_absolute_error(y, train_preds))


🔹 Fold 0
🔹 Fold 1
🔹 Fold 2
🔹 Fold 3
🔹 Fold 4
Base OOF MAE: 0.16194219425912912


In [42]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold

# 1. Pre-normalize once to avoid doing it repeatedly inside the loop
emb_norm = normalize(train_embs)
K = 30

# --- Part 1: Fast Global Train Correction ---
knn = NearestNeighbors(n_neighbors=K + 1, metric="cosine", n_jobs=-1) # Use n_jobs=-1
knn.fit(emb_norm)
_, idxs = knn.kneighbors(emb_norm)

# Vectorized mean: idxs[:, 1:] is (N, K). 
# residuals[idxs[:, 1:]] creates a (N, K) matrix of neighbor residuals.
res_corr = residuals[idxs[:, 1:]].mean(axis=1) 

train_preds_knn = np.clip(train_preds + res_corr, 0, 1)
print(f"🔥 KNN-corrected TRAIN MAE: {mean_absolute_error(y, train_preds_knn):.5f}")

# --- Part 2: Fast OOF K-Fold Correction ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_knn = np.zeros(len(y))

for tr, va in kf.split(train_embs):
    # Use the pre-normalized embeddings
    emb_tr, emb_va = emb_norm[tr], emb_norm[va]
    res_tr = residuals[tr]
    
    knn = NearestNeighbors(n_neighbors=K, metric="cosine", n_jobs=-1)
    knn.fit(emb_tr)
    
    # We only need indices, not distances
    _, idxs = knn.kneighbors(emb_va)
    
    # VECTORIZED: map residuals of train set to indices found for val set
    corr = res_tr[idxs].mean(axis=1) 
    
    oof_knn[va] = np.clip(train_preds[va] + corr, 0, 1)

print(f"🔥 SAFE KNN-CORRECTED OOF MAE: {mean_absolute_error(y, oof_knn):.5f}")

🔥 KNN-corrected TRAIN MAE: 0.17309
🔥 SAFE KNN-CORRECTED OOF MAE: 0.19457


In [43]:
import numpy as np
from collections import Counter, defaultdict

def routing_features(row):
    tokens = [t for t,m in zip(row["input_ids"], row["attention_mask"]) if m]
    L = len(tokens)

    # length
    length = L
    pos = defaultdict(list)
    for i, t in enumerate(tokens):
        pos[t].append(i / L)
    rpc = np.mean([np.var(v) for v in pos.values() if len(v) >= 2]) if L else 1.0
    # entropy
    cnt = Counter(tokens)
    freqs = np.array(list(cnt.values())) / max(L,1)
    entropy = -np.sum(freqs * np.log(freqs + 1e-8)) / np.log(max(L,2))

    # constraint score
    constraint = sum(v >= 3 for v in cnt.values())

    return np.array([length, entropy, constraint,rpc], dtype=np.float32)

X_route = np.vstack([routing_features(r) for r in train_data])


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [44]:
# Length bins
len_bins = np.quantile(X_route[:,0], [0, 0.33, 0.66, 1.0])

# Entropy bins
ent_bins = np.quantile(X_route[:,1], [0, 0.33, 0.66, 1.0])

# Constraint bins (manual, skewed)
con_bins = [0, 1, 3, 1e9]
def assign_bin(x, bins):
    return np.digitize(x, bins[1:-1])

len_bin = np.array([assign_bin(v, len_bins) for v in X_route[:,0]])
ent_bin = np.array([assign_bin(v, ent_bins) for v in X_route[:,1]])
con_bin = np.array([assign_bin(v, con_bins) for v in X_route[:,2]])
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from lightgbm import LGBMRegressor

eps = 1e-4
y_logit = np.log((y + eps) / (1 - y + eps))

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_final = np.zeros(len(y))

for fold, (tr, va) in enumerate(kf.split(X_oof_dev)):
    print(f"\n🧠 Fold {fold}")

    preds = []

    for bin_name, bin_assign in [
        ("LEN", len_bin),
        ("ENT", ent_bin),
        ("CON", con_bin),
    ]:
        print(f"  ▶ Expert: {bin_name}")

        pred_bin = np.zeros(len(va))

        for b in np.unique(bin_assign):
            tr_idx = tr[bin_assign[tr] == b]
            va_idx = va[bin_assign[va] == b]

            if len(tr_idx) < 500 or len(va_idx) == 0:
                continue

            model = LGBMRegressor(
                objective="regression_l1",
                n_estimators=1200,
                learning_rate=0.01,
                max_depth=4,
                num_leaves=24,
                subsample=0.7,
                colsample_bytree=0.7,
                reg_alpha=0.5,
                reg_lambda=1.0,
                random_state=fold,
                verbosity=-1
            )

            model.fit(X_oof_dev.iloc[tr_idx], y_logit[tr_idx])

            p = model.predict(X_oof_dev.iloc[va_idx])
            p = 1 / (1 + np.exp(-p))

            pred_bin[np.isin(va, va_idx)] = p

        preds.append(pred_bin)

    # average experts
    preds = np.stack(preds)
    oof_final[va] = preds.mean(axis=0)

print("\n🔥 HARD-BIN MoE SAFE MAE:",
      mean_absolute_error(y, oof_final))



🧠 Fold 0
  ▶ Expert: LEN
  ▶ Expert: ENT
  ▶ Expert: CON

🧠 Fold 1
  ▶ Expert: LEN
  ▶ Expert: ENT
  ▶ Expert: CON

🧠 Fold 2
  ▶ Expert: LEN
  ▶ Expert: ENT
  ▶ Expert: CON

🧠 Fold 3
  ▶ Expert: LEN
  ▶ Expert: ENT
  ▶ Expert: CON

🧠 Fold 4
  ▶ Expert: LEN
  ▶ Expert: ENT
  ▶ Expert: CON

🔥 HARD-BIN MoE SAFE MAE: 0.1524820485155164


In [48]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning) # Optional: Global silence

In [49]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor
import numpy as np

eps = 1e-4
y_logit = np.log((y + eps) / (1 - y + eps))

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_final = np.zeros(len(y))

for fold, (tr, va) in enumerate(kf.split(X_oof_dev)):
    print(f"\n🧠 Fold {fold}")

    expert_preds = []
    expert_masks = []

    # ======================
    # Train experts
    # ======================
    for name, bin_assign in [
        ("LEN", len_bin),
        ("ENT", ent_bin),
        ("CON", con_bin),
    ]:
        print(f"  ▶ Training expert {name}")
        pred = np.zeros(len(va))
        mask = np.zeros(len(va))

        for b in np.unique(bin_assign):
            tr_idx = tr[bin_assign[tr] == b]
            va_idx = va[bin_assign[va] == b]

            if len(tr_idx) < 500 or len(va_idx) == 0:
                continue

            model = LGBMRegressor(
                objective="regression_l1",
                n_estimators=1000,
                learning_rate=0.01,
                max_depth=4,
                num_leaves=24,
                subsample=0.7,
                colsample_bytree=0.7,
                reg_alpha=0.5,
                reg_lambda=1.0,
                random_state=fold,
                verbosity=-1
            )

            # Use .values to avoid feature name warnings
            model.fit(X_oof_dev.iloc[tr_idx].values, y_logit[tr_idx])
            p = model.predict(X_oof_dev.iloc[va_idx].values)
            p = 1 / (1 + np.exp(-p))

            # Logic to map local fold indices to global validation indices
            local_va_mask = np.isin(va, va_idx)
            pred[local_va_mask] = p
            mask[local_va_mask] = 1

        expert_preds.append(pred)
        expert_masks.append(mask)

    # expert_preds and expert_masks shape: (len(va), 3)
    expert_preds = np.stack(expert_preds, axis=1)
    expert_masks = np.stack(expert_masks, axis=1)

    # ======================
    # Train GATE
    # ======================
    print("  ▶ Training gate")

    # MultiOutputRegressor allows the gate to predict 3 weights (one per expert)
    gate = MultiOutputRegressor(LGBMRegressor(
        objective="regression",
        n_estimators=500,
        learning_rate=0.05,
        max_depth=3,
        num_leaves=16,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=fold,
        verbosity=-1
    ))

    # We train the gate to target 'y'. MultiOutput targets shape (N, 3)
    gate.fit(X_route[tr], np.tile(y[tr], (3, 1)).T)
    w = gate.predict(X_route[va]) # Now shape is (len(va), 3)

    # softmax weights applied across the expert dimension
    # expert_masks (len(va), 3) ensures we only weight experts that actually ran
    w_exp = np.exp(w) * expert_masks
    w_final = w_exp / (w_exp.sum(axis=1, keepdims=True) + 1e-6)

    # final prediction: weighted sum of the 3 experts
    oof_final[va] = (expert_preds * w_final).sum(axis=1)

print("\n🔥 STACKED GATED MoE SAFE MAE:",
      mean_absolute_error(y, oof_final))


🧠 Fold 0
  ▶ Training expert LEN
  ▶ Training expert ENT
  ▶ Training expert CON
  ▶ Training gate

🧠 Fold 1
  ▶ Training expert LEN
  ▶ Training expert ENT
  ▶ Training expert CON
  ▶ Training gate

🧠 Fold 2
  ▶ Training expert LEN
  ▶ Training expert ENT
  ▶ Training expert CON
  ▶ Training gate

🧠 Fold 3
  ▶ Training expert LEN
  ▶ Training expert ENT
  ▶ Training expert CON
  ▶ Training gate

🧠 Fold 4
  ▶ Training expert LEN
  ▶ Training expert ENT
  ▶ Training expert CON
  ▶ Training gate

🔥 STACKED GATED MoE SAFE MAE: 0.15244741051224595
